# Дискреционная политика безопасности

In [ ]:
import random

### Класс пользователя

In [ ]:
class User:
    # Все возможные комбинации прав
    RIGHTS_OPTIONS = [
        set(),
        ("read"),
        ("write"),
        ("grant"),
        ("read", "write"),
        ("read", "grant"),
        ("write", "grant"),
        ("read", "write", "grant")
    ]
    
    def __init__(self, username, objects):
        self.username = username
        self.objects = objects  # список объектов (например, файлов)
        self.access_rights = {}  # матрица доступа: ключ — объект, значение — набор прав
        self.fill_access_rights()
        
    def fill_access_rights(self):
        """
        Программно заполняет матрицу доступа для каждого объекта.
        Если пользователь – admin, то устанавливаются все права.
        Для остальных выбирается случайная комбинация из всех возможных вариантов.
        """
        for obj in self.objects:
            if self.username == "admin":
                self.access_rights[obj] = {"read", "write", "grant"}
            else:
                self.access_rights[obj] = random.choice(self.RIGHTS_OPTIONS)
    
    def get_rights_string(self, obj):
        """
        Преобразует набор прав для объекта в строковое представление.
        Если права полные – выводится "Полные права", если пустой набор – "Запрет",
        иначе права выводятся в порядке: Чтение, Запись, Передача прав.
        """
        rights = self.access_rights.get(obj, set())
        if rights == {"read", "write", "grant"}:
            return "Полные права"
        if not rights:
            return "Запрет"
        mapping = {"read": "Чтение", "write": "Запись", "grant": "Передача прав"}
        rights_ordered = [mapping[r] for r in ["read", "write", "grant"] if r in rights]
        return ", ".join(rights_ordered)
    
    def display_rights(self):
        """
        Выводит перечень прав пользователя для всех объектов.
        """
        print("Перечень ваших прав:")
        for idx, obj in enumerate(self.objects, start=1):
            print(f"{idx}) {obj}: {self.get_rights_string(obj)}")

    def has_permission(self, obj, permission):
        """
        Проверяет, имеет ли пользователь для объекта заданное право.
        """
        return permission in self.access_rights.get(obj, set())

### Работа с пользователями

In [ ]:
# Список объектов доступа (например, файлов)
objects = ["file1", "file2", "file3"]
# Начальный список пользователей
initial_usernames = ["user1", "user2", "user3", "user4", "user5", "user6", "user7", "admin"]

# Создаем словарь пользователей: ключ – имя, значение – экземпляр класса User
users = {username: User(username, objects) for username in initial_usernames}

# Запрос идентификатора пользователя
user_id = input("Введите идентификатор пользователя: ").strip()
if user_id not in users:
print("Пользователь не найден. Завершаю работу.")
return

current_user = users[user_id]
print(f"User: {current_user.username}")
print("Идентификация прошла успешно, добро пожаловать в систему")
current_user.display_rights()

# Цикл ожидания команд
while True:
command = input("Жду ваших указаний > ").strip().lower()

if command == "quit":
    print(f"Работа пользователя {current_user.username} завершена. До свидания.")
    break
elif command == "read" or command == "write":
    try:
        obj_index = int(input("Над каким объектом производится операция? ")) - 1
        if obj_index < 0 or obj_index >= len(objects):
            print("Некорректный номер объекта.")
            continue
        obj = objects[obj_index]
        if current_user.has_permission(obj, command):
            print("Операция прошла успешно")
        else:
            print("Отказ в выполнении операции. У Вас нет прав для ее осуществления")
    except ValueError:
        print("Некорректный ввод. Ожидался номер объекта.")

elif command == "grant":
    try:
        obj_index = int(input("Право на какой объект передается? ")) - 1
        if obj_index < 0 or obj_index >= len(objects):
            print("Некорректный номер объекта.")
            continue
        obj = objects[obj_index]
        if not current_user.has_permission(obj, "grant"):
            print("Отказ в выполнении операции. У Вас нет прав для ее осуществления")
            continue
        right_to_grant = input("Какое право передается? ").strip().lower()
        if right_to_grant not in ["read", "write", "grant"]:
            print("Некорректное право. Допустимы: read, write, grant.")
            continue
        target_user = input("Какому пользователю передается право? ").strip()
        if target_user not in users:
            print("Пользователь не найден.")
            continue
        # Передаем право: добавляем его в набор прав целевого пользователя для данного объекта
        users[target_user].access_rights[obj].add(right_to_grant)
        print("Операция прошла успешно")
    except ValueError:
        print("Некорректный ввод. Ожидался номер объекта.")

elif command == "create":
    # Возможность создания нового пользователя доступна только администратору
    if current_user.username != "admin":
        print("Отказ: у Вас нет прав для создания новых пользователей")
        continue
    new_username = input("Введите имя нового пользователя: ").strip()
    if new_username in users:
        print("Пользователь с таким именем уже существует.")
    else:
        users[new_username] = User(new_username, objects)
        print(f"Пользователь '{new_username}' успешно создан.")

elif command == "delete":
    # Возможность удаления пользователя доступна только администратору
    if current_user.username != "admin":
        print("Отказ: у Вас нет прав для удаления пользователей")
        continue
    del_username = input("Введите имя пользователя, которого необходимо удалить: ").strip()
    if del_username not in users:
        print("Пользователь не найден.")
    elif del_username == "admin":
        print("Нельзя удалить администратора.")
    else:
        del users[del_username]
        print(f"Пользователь '{del_username}' успешно удалён.")
else:
    print("Неверная команда. Попробуйте снова.")